## XGBoost Hyperparameter Tuning

In this notebook, we will perform hyperparameter tuning on our XGBoost model using Optuna.

## Set up
Optuna-integration[xgboost] is an xgboost integreation for optuna that serves as a specialized library specifically for our purpose.

In [1]:
# install relavent libraries - run once per environment
# !pip install optuna
# !pip install optuna-integration[xgboost]

In [2]:
# import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import optuna

## Load, Process, and Prepare Data

In [3]:
# load data
X_train = pd.read_csv("train_transaction.csv")
X_val = pd.read_csv("val_transaction.csv")

# Remove unnecessary columns left over from cleaning
X_train.drop(['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0'], axis=1, inplace=True)
X_val.drop(['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0'], axis=1, inplace=True)

# Split target column from main data
y_train = X_train['isFraud']
X_train.drop(['isFraud'], axis=1, inplace=True)

y_val = X_val['isFraud']
X_val.drop(['isFraud'], axis=1, inplace=True)

In [4]:
# Remove all email columns
X_train_v2 = X_train[X_train.columns.drop(list(X_train.filter(regex='email')))]
X_val_v2 = X_val[X_val.columns.drop(list(X_val.filter(regex='email')))]

# make xgb datasets
train_data = xgb.DMatrix(X_train_v2, label=y_train, enable_categorical=True)
val_data = xgb.DMatrix(X_val_v2, label=y_val, enable_categorical=True)


## Set Up Hyperparameter Tuning

Pruning allows us to exit unpromising iterations early to preserve compute.

In [32]:
# create function to set up model for each trial
def objective(trial):
    # set up data
    train_data = xgb.DMatrix(X_train_v2, label=y_train, enable_categorical=True)
    val_data = xgb.DMatrix(X_val_v2, label=y_val, enable_categorical=True)

    # train model
    xgb_params = {
        'max_leaves': trial.suggest_int('max_leaves', 250, 400),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 100, log=True),
        'objective': 'binary:logistic',
        'max_depth': trial.suggest_int('max_depth', 3, 13),
        'eta': trial.suggest_float("eta", .001, 1.0, log=True),
        'gamma': trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        'tree_method': 'hist',
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'seed': 11,
        'eval_metric': 'auc',
        'verbosity': 0,
        'reg_alpha': trial.suggest_float("alpha", 1e-6, 10.0, log=True),
        'reg_lambda': trial.suggest_float("lambda", 0.1, 10.0, log=True),
        'grow_policy': trial.suggest_categorical('grow_policy', ["depthwise", "lossguide"]),
    }

    # leaf pruning - exit unpromising models early
    prune = optuna.integration.XGBoostPruningCallback(trial, "validation-auc")
    best = xgb.train(
        xgb_params,
        train_data,
        num_boost_round=10,
        evals=[(val_data, "validation")],
        early_stopping_rounds=50,
        callbacks=[prune],
        verbose_eval=False,
    )
    preds = best.predict(val_data, iteration_range=(0, best.best_iteration + 1))
    auc = roc_auc_score(y_val, preds)
    return auc

## Perform Hyperparameter Tuning

In [33]:
# run study to maximize score
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),  # reproducible search
)

# find the best hyperparameters through testing multiple trials
study.optimize(objective, n_trials=250)
# print best trial
print(study.best_trial)

[I 2026-08-14 20:53:58,745] A new study created in memory with name: no-name-75854000-c70b-4364-9b96-ee017782ba95
[I 2026-08-14 20:54:02,869] Trial 0 finished with value: 0.8796669908801659 and parameters: {'max_leaves': 306, 'min_child_weight': 77, 'max_depth': 11, 'eta': 0.06251373574521749, 'gamma': 1.77071686435378e-07, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'alpha': 1.156732719914599, 'lambda': 1.5930522616241019, 'grow_policy': 'depthwise'}. Best is trial 0 with value: 0.8796669908801659.
[I 2026-08-14 20:54:07,808] Trial 1 finished with value: 0.8436375569242426 and parameters: {'max_leaves': 396, 'min_child_weight': 41, 'max_depth': 5, 'eta': 0.0035113563139704067, 'gamma': 2.9324868872723725e-07, 'subsample': 0.6521211214797689, 'colsample_bytree': 0.762378215816119, 'alpha': 0.0010558813779064837, 'lambda': 0.38234752246751863, 'grow_policy': 'depthwise'}. Best is trial 0 with value: 0.8796669908801659.
[I 2026-08-14 20:54:12,184] Trial 2 fin

FrozenTrial(number=231, state=<TrialState.COMPLETE: 1>, values=[0.9354330389978354], datetime_start=datetime.datetime(2026, 8, 14, 21, 11, 55, 859699), datetime_complete=datetime.datetime(2026, 8, 14, 21, 12, 4, 800757), params={'max_leaves': 374, 'min_child_weight': 1, 'max_depth': 13, 'eta': 0.3727634685843487, 'gamma': 0.00014121726480517213, 'subsample': 0.9798380685518304, 'colsample_bytree': 0.6511299034135586, 'alpha': 0.006078931276328701, 'lambda': 9.641597078203631, 'grow_policy': 'lossguide'}, user_attrs={}, system_attrs={}, intermediate_values={0: 0.8827384397632106, 1: 0.904472858116502, 2: 0.9112241860579869, 3: 0.9170741291461184, 4: 0.9201356597308127, 5: 0.9253850920525963, 6: 0.9291566027349527, 7: 0.9319992771508626, 8: 0.9333095395703807, 9: 0.9354330389978354}, distributions={'max_leaves': IntDistribution(high=400, log=False, low=250, step=1), 'min_child_weight': IntDistribution(high=100, log=True, low=1, step=1), 'max_depth': IntDistribution(high=13, log=False, lo

In [34]:
# print out best parameters and the best model's accuracy incase of a crash/enviroment shutdown
print(f"Best parameters: {study.best_params}")
print(f"Best accuracy: {study.best_value}")

Best parameters: {'max_leaves': 374, 'min_child_weight': 1, 'max_depth': 13, 'eta': 0.3727634685843487, 'gamma': 0.00014121726480517213, 'subsample': 0.9798380685518304, 'colsample_bytree': 0.6511299034135586, 'alpha': 0.006078931276328701, 'lambda': 9.641597078203631, 'grow_policy': 'lossguide'}
Best accuracy: 0.9354330389978354


In the final model, we will be using the hyperparameters found through the tuning process.